In [48]:
import numpy as np
import pandas as pd

In [49]:
from datasets import load_dataset
ds1 = load_dataset("KisanVaani/agriculture-qa-english-only")
ds2 = load_dataset("argilla/farming")
ds3 = load_dataset("shchoi83/agriQA")

Repo card metadata block was not found. Setting CardData to empty.


In [50]:
ds1

DatasetDict({
    train: Dataset({
        features: ['question', 'answers'],
        num_rows: 22615
    })
})

In [51]:
ds2

DatasetDict({
    train: Dataset({
        features: ['id', 'instruction', 'response'],
        num_rows: 1695
    })
})

In [52]:
ds3

DatasetDict({
    train: Dataset({
        features: ['questions', 'answers', 'text'],
        num_rows: 174930
    })
})

In [53]:
# Convert to DataFrame
df1 = pd.DataFrame(ds1["train"])
df2 = pd.DataFrame(ds2["train"])
df3 = pd.DataFrame(ds3["train"])

print("Original sizes:")
print("KisanVaani:", len(df1))
print("Argilla:", len(df2))
print("agriQA:", len(df3))

Original sizes:
KisanVaani: 22615
Argilla: 1695
agriQA: 174930


In [54]:
df1 = df1.rename(columns={
    "question": "question",
    "answers": "answer"
})

df1["source"] = "kisanvaani"
df1 = df1[["question", "answer", "source"]]

In [55]:
df2 = df2.rename(columns={
    "instruction": "question",
    "response": "answer"
})

df2["source"] = "argilla"
df2 = df2[["question", "answer", "source"]]


In [56]:
df3 = df3.rename(columns={
    "questions": "question",
    "answers": "answer"
})

df3["source"] = "agriqa"
df3 = df3[["question", "answer", "source"]]


In [57]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    
    # Remove boilerplate formatting
    text = re.sub(r"Below are questions.*?###A:", "", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

# Apply cleaning
df3["question"] = df3["question"].apply(clean_text)
df3["answer"] = df3["answer"].apply(clean_text)

# Remove short answers (less than 15 characters)
df3 = df3[df3["answer"].str.len() > 15]

# Remove duplicates
df3 = df3.drop_duplicates(subset=["question", "answer"])

print("Filtered agriQA size:", len(df3))


Filtered agriQA size: 134771


In [58]:
keywords = [
    "rice", "paddy", "fertilizer", "soil", "irrigation", "harvesting", "sowing"
    "pest", "disease", "milk", "crop", "yield", "sun", "climate", "sunlight",
    "seed", "manure", "weed", "livestock", "rain", "rainfall", "water", "profit", "drought", "dryness"
]

pattern = "|".join(keywords)

dff3 = df3[
    df3["question"].str.contains(pattern, case=False, na=False) |
    df3["answer"].str.contains(pattern, case=False, na=False)
]

print("Keyword filtered agriQA size:", len(dff3))


Keyword filtered agriQA size: 88359


In [59]:
for df in [df1, df2]:
    df["question"] = df["question"].apply(clean_text)
    df["answer"] = df["answer"].apply(clean_text)

df1 = df1.drop_duplicates(subset=["question", "answer"])
df2 = df2.drop_duplicates(subset=["question", "answer"])


In [60]:
print("Before cleaning:", len(pd.DataFrame(ds1["train"])))

temp_df = pd.DataFrame(ds1["train"])
temp_df = temp_df.rename(columns={"question": "question", "answers": "answer"})

print("Unique before cleaning:", temp_df.drop_duplicates(subset=["question", "answer"]).shape[0])


Before cleaning: 22615
Unique before cleaning: 2331


In [61]:
df1_test = temp_df.copy()

df1_test["question"] = df1_test["question"].apply(clean_text)
df1_test["answer"] = df1_test["answer"].apply(clean_text)

print("Unique after cleaning:", df1_test.drop_duplicates(subset=["question", "answer"]).shape[0])


Unique after cleaning: 2331


In [66]:
final_df = pd.concat([df1, df2, dff3], ignore_index=True)

# Remove global duplicates
final_df = final_df.drop_duplicates(subset=["question", "answer"])

print("Final dataset size:", len(final_df))


Final dataset size: 92385


In [67]:
from sentence_transformers import SentenceTransformer
import numpy as np

model_embed = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus = final_df["question"].tolist()

embeddings = model_embed.encode(
    corpus,
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64
)

print("Embedding shape:", embeddings.shape)


2026-02-19 15:14:12.678764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771514052.869848      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771514052.923667      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771514053.375744      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771514053.375789      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771514053.375792      55 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1444 [00:00<?, ?it/s]

Embedding shape: (92385, 384)


In [69]:
!pip install faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.9 MB/s eta 0:00:00:00:0100:01


In [70]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 92385


In [71]:
def retrieve(query, top_k=5):
    query_embedding = model_embed.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = final_df.iloc[indices[0]]

    return results


In [73]:
tt = retrieve("How to control aphid infestation in rice?")

In [78]:
tt

,question,answer,source
39409,asking about the control measure to control ap...,suggested to spray nuvacron 40ec@2 mililitre/l...,agriqa
4307,asking about to controll of aphids in rice.,suggested him to spray durseban @ 2ml/liter of...,agriqa
45007,asking about the control of aphid in rice.,recommended to apply tricel 20 ec @ 2ml per li...,agriqa
51954,asking about the control measure to control ap...,suggested to apply furadon3g@3gram/squar meter.,agriqa
58559,problem of aphids in boro rice.,advised him to apply rogor 35 ec @ 2 ml / lite...,agriqa


In [77]:
tt.iloc[0]["answer"]

'suggested to spray nuvacron 40ec@2 mililitre/litre of water.'

We selected Qwen1.8B as the generative backbone because it provides strong reasoning quality while remaining computationally efficient for single-GPU Kaggle environments. Larger models such as Mistral-7B were considered but were not optimal due to VRAM constraints and inference latency.

In [79]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen1.5-1.8B-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_gen = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

In [88]:
def generate_answer(query, top_k=5):
    # Step 1: Retrieve relevant entries
    retrieved = retrieve(query, top_k=top_k)

    # Step 2: Build numbered retrieved knowledge
    knowledge_block = ""
    for idx, row in enumerate(retrieved.itertuples(), start=1):
        knowledge_block += f"{idx}. {row.answer.strip()}\n"

    # Step 3: Construct strict grounding prompt
    prompt = f"""
You are an agricultural advisory system.

Below are retrieved knowledge entries numbered 1 to {len(retrieved)}.

You must ONLY use the numbered knowledge entries to construct your answer.

Rules:
- Do NOT introduce new techniques.
- Do NOT add external explanations.
- If required information is missing, say:
  "Detail not available in retrieved knowledge."

Retrieved Knowledge:
{knowledge_block}

Farmer Question:
{query}

Provide a clear, structured, practical answer based strictly on the retrieved knowledge.
Answer:
"""

    # Step 4: Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Step 5: Generate
    output = model_gen.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.1,          # lower = more controlled
        do_sample=False           # deterministic output
    )

    # Step 6: Decode only generated portion
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()


In [97]:
t = retrieve("How to manage paddy crop during drought conditions?")


In [101]:
t

,question,answer,source
73018,facing drought problem in sali paddy,adviced to apply irrigation immediately,agriqa
845,what water managing techniques can a farmer us...,mulching,kisanvaani
735,Ways farmers can guard against drought.,Crop rotation,kisanvaani
843,what irrigation techniques can farmers use to ...,drip or sprinkler irrigation,kisanvaani
841,name one way a farmer can guard against drought?,Crop Selection: Farmers should select crops th...,kisanvaani


In [100]:
t.iloc[1]

question    what water managing techniques can a farmer us...
answer                                               mulching
source                                             kisanvaani
Name: 845, dtype: object

✔ "What is the fertilizer dose for cabbage?"
✔ "How to control late blight in potato?"
✔ "Fertilizer dose for semi dwarf rice?"
✔ "Control measure for aphids in rice?"

In [95]:
print(generate_answer("What is the fertilizer dose for cabbage?"))

The fertilizer dose for cabbage can be recommended using the following knowledge entries:

1. Urea: The recommended amount of urea for cabbage is 36 kg/bigha in a split dose, which means applying it in two separate applications. This dosage is spread evenly across the cabbage plants and helps to promote root growth, enhance nutrient uptake, and prevent wilting.

2. Ssp (Supplementary Spraying): To provide additional nitrogen to the cabbage plants, a supplement sprayer can be used at a rate of 50 kg/bigha. This application should be applied immediately after the basal dose of urea, ensuring that all cabbage leaves receive adequate nitrogen support.

3. Mop: A mop can also be added to the basal dose of urea to help with water retention and reduce surface water evaporation. The mop should be applied at a rate of 13 kg/bigha, ensuring that the entire cabbage bed receives sufficient moisture.

4. Borax: Borax is a mineral that can be used to improve soil fertility by increasing pH levels an

In [96]:
print(generate_answer("How to control late blight in potato?"))

To effectively control late blight in potatoes, farmers can follow these steps:

1. Identify the affected plants: Late blight is a fungal disease that affects the leaves and stems of potatoes. The first step is to identify the affected plants by inspecting their leaves for yellowing, discoloration, or necrosis. This will help determine if the disease has already spread to other parts of the crop.

2. Apply fungicides: Once the affected plants have been identified, the next step is to apply appropriate fungicide treatments. The choice of fungicide depends on the type of late blight present and the severity of the infection. Some common fungicides used for potato late blight include spray ridomil (Mz 72), spray indofil (Mz 45), and spray ridomil gold (Mz 80). These products contain active ingredients such as pyrimidinones, tricyclic trisols, and azole compounds, which target the fungus's cell walls and inhibit its growth.

3. Timing of application: It is essential to apply fungicide trea

In [ ]:
print(generate_answer("What is the fertilizer dose for semi-dwarf rice variety?"))

In [89]:
print(generate_answer("How to increase yield in paddy crop?"))

To increase the yield in paddy crops, several steps can be taken to optimize soil fertility and promote healthy plant growth. Here's a step-by-step guide based on the provided knowledge:

1. Soil Fertility Improvement:
   - Use Organic Matter: Incorporate organic matter into the soil, such as compost, manure, or leaf litter. This helps to enrich the soil structure, increase its water-holding capacity, and provide essential nutrients for plants. Organic matter also promotes microbial activity, which aids in nutrient cycling and disease suppression.
   - Maintain Proper Soil pH: Paddy crops thrive in slightly acidic (6.0-7.0) soil conditions. Adjusting the soil pH to this range will help prevent nutrient deficiencies and promote healthy root development. You can use a soil test kit to determine the current pH level and adjust it accordingly.
   - Apply Organic NPK Solutions: Organic nitrogen (N), phosphorus (P), and potassium (K) fertilizers can be applied in the form of granular or liqu

In [90]:
print(generate_answer("What is the fertilizer dose for semi-dwarf rice variety?"))

The fertilizer dose for semi-dwarf rice variety can be determined by considering the following factors:

1. Urea requirement: The recommended fertilizer dose for semi-dwarf rice varieties typically includes urea and sulfur phosphorus (SP). According to the provided knowledge entry, the suggested dosage for urea is 12 kg/bigha, with three parts:

(i) 2 days before transplanting at the final field preparation: This part of the recommendation suggests applying 2 kg of urea per day for 2 days before transplanting. So, the total urea dose for this stage would be 2 kg/day x 2 days = 4 kg.

(ii) 1 month after transplanting: After transplanting, the urea dose should be applied immediately to promote root growth and prevent nutrient leaching from the soil. Therefore, the total urea dose for this stage would be 4 kg + 4 kg = 8 kg.

3. Lime application: Lime is often used as a pre-emergent herbicide to suppress weeds and improve soil structure. Lime has a pH-adjusting effect, which helps maintain

In [91]:
print(generate_answer("How to manage paddy crop during drought conditions?"))

During drought conditions, managing paddy crops requires careful planning and implementation of various strategies to ensure optimal yields while minimizing water loss. Here's a step-by-step guide for farmers to manage paddy crops effectively:

1. Water conservation: The first step is to conserve water by implementing efficient irrigation practices. Drip or sprinkler irrigation systems deliver water directly to the roots of plants, reducing evaporation and runoff. This method ensures that water reaches the plant's root zone without being lost through evaporation or surface runoff. Additionally, using mulch around the plants can help retain moisture in the soil, reducing evaporation rates. Mulching helps regulate soil temperature, which can be crucial during dry periods when transpiration rates are high.

2. Crop selection: Selecting crops that are adapted to local climate conditions and can tolerate drought conditions is essential. Paddy crops like maize, sorghum, and millet are known 

In [92]:
print(generate_answer("How to prevent crop damage due to excess rainfall?"))

To prevent crop damage due to excessive rainfall, farmers can implement several strategies that take into account the specific needs of their crops and the local environment:

1. Choose appropriate varieties: Selecting rain-resistant varieties of rice, such as jal kowari and jal shree, can help reduce the risk of crop damage from heavy rainfall. These varieties have been bred to withstand flooding and maintain good yields even under waterlogging conditions.

2. Optimize irrigation practices: Implementing efficient irrigation systems can minimize water loss during rainfall events. This includes using drip irrigation, which delivers water directly to plant roots, reducing evaporation and runoff. Additionally, using mulch around plants can help retain moisture in the soil, preventing it from evaporating quickly during rainfall.

3. Maintain proper drainage: Ensuring adequate drainage around fields and paddocks can help prevent waterlogging and promote healthy root growth. This involves cr

In [93]:
print(generate_answer("How to increase profit in rice farming?"))

To increase profits in rice farming, it is essential to follow proper cultivation practices, apply appropriate fertilizers, and manage pests effectively. Here's a step-by-step guide:

1. Follow Proper Cultivation Practices:
   - Planting: Choose the right variety of rice that suits the local climate and soil conditions. Ensure that the seedlings are well-watered and receive adequate sunlight during germination.
   - Soil Management: Opt for organic or natural fertilizers like compost, manure, or fish emulsion to enrich the soil with nutrients. Avoid using chemical fertilizers, which can harm the environment and reduce crop yields.
   - Irrigation: Water the crops regularly, especially during dry spells, to prevent nutrient loss due to evaporation. Use drip irrigation systems or sprinklers to deliver water directly to the roots, reducing water waste.
   - Crop Rotation: Rotate crops between different fields to maintain soil fertility and prevent pest buildup. This helps in preventing di

92k QA corpus
        ↓
MiniLM embeddings (384-dim)
        ↓
FAISS index
        ↓
User query embedding
        ↓
Top-K retrieval
        ↓
Context assembly
        ↓
Qwen 1.8B generation
        ↓
Final agronomic answer


“The system integrates a pretrained decoder-based LLM with a domain-specific vector retrieval mechanism. Instead of modifying model weights, knowledge grounding is achieved through retrieval augmentation. This aligns with modern large-scale GenAI system design.”

In [6]:
data = ds1["train"]

print("Total rows:", len(data))
print(data[0])

Total rows: 22615
{'question': 'why is crop rotation important in farming?', 'answers': 'This helps to prevent soil erosion and depletion, and can also help to control pests and diseases'}


In [7]:
data = data.select(range(15000))

In [8]:
questions = data["question"]
answers = data["answers"]

print("Sample Question:", questions[0])
print("Sample Answer:", answers[0])


Sample Question: why is crop rotation important in farming?
Sample Answer: This helps to prevent soil erosion and depletion, and can also help to control pests and diseases


In [9]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


2026-02-18 11:21:17.300125: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771413677.629693      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771413677.733956      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771413678.595055      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771413678.595107      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771413678.595110      55 computation_placer.cc:177] computation placer alr

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
print("Creating embeddings...")
question_embeddings = model.encode(
    questions,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", question_embeddings.shape)


Creating embeddings...


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Embedding shape: (15000, 384)


In [11]:
np.save("agri_embeddings.npy", question_embeddings)

In [12]:
# data = {
#     "questions": questions,
#     "answers": answers
# }

# with open("agro_qa_data.pkl", "wb") as f:
#     pickle.dump(data, f)

# print("✅ Saved embeddings and QA data successfully")


In [15]:
import pickle


In [16]:
questions = list(data["question"])
answers = list(data["answers"])

clean_data = {
    "questions": questions,
    "answers": answers
}

with open("agro_qa_data.pkl", "wb") as f:
    pickle.dump(clean_data, f)

print("✅ Saved clean QA data successfully")


✅ Saved clean QA data successfully


In [ ]:
# import numpy as np

# print("Creating embeddings...")
# question_embeddings = embed_model.encode(
#     questions,
#     convert_to_numpy=True,
#     show_progress_bar=True
# )

# print("Embedding shape:", question_embeddings.shape)


In [40]:
# !pip install sentence-transformers faiss-cpu datasets -q


In [41]:
# import faiss

# dimension = question_embeddings.shape[1]

# index = faiss.IndexFlatL2(dimension)
# index.add(question_embeddings)

# print("FAISS index built")
# print("Total vectors in index:", index.ntotal)


FAISS index built
Total vectors in index: 14000


In [53]:
# def retrieve_answer(query, top_k=3):
#     query_embedding = embed_model.encode([query], convert_to_numpy=True)
    
#     distances, indices = index.search(query_embedding, top_k)
    
#     results = []
#     for idx in indices[0]:
#         results.append({
#             "question": questions[idx],
#             "answer": answers[idx]
#         })
    
#     return results


# # Test
# query = "How can I increase my crop yield?"
# results = retrieve_answer(query)

# for r in results:
#     print("\nQ:", r["question"])
#     print("A:", r["answer"])


In [43]:
# import pickle

# # Save FAISS index
# faiss.write_index(index, "agri_qa_faiss.index")

# # Save questions & answers
# with open("agri_qa_data.pkl", "wb") as f:
#     pickle.dump({
#         "questions": questions,
#         "answers": answers
#     }, f)

# print("✅ Saved FAISS index and data successfully")


✅ Saved FAISS index and data successfully


In [54]:
# SentenceTransformer("all-MiniLM-L6-v2")
